# Field weights — the same search budget for both arms

Two questions, one experiment.

**1. Does `description` carry retrieval signal at ANY weight?** The four description variants came
in within 0.5% of each other, but that test was underpowered by construction: `reviews` carries
w=0.5, i.e. w²=0.25 of a total 3.5, so the field is only **7.1% of an item's squared norm**. Even a
perfect-versus-useless swing there could barely move the objective. Sweeping the weight — including
**w=0**, which deletes the block outright — separates "representation doesn't matter" from "the
field doesn't matter".

**2. Is the TF-IDF baseline's 20.7% lead a property of the representation, or of its weights?**
CBHCF tuned λ and nothing else; `content.DEFAULT_WEIGHTS` are reasoned defaults that were never
searched. Tuning only Intervention A would hand it a budget the baseline never had — the exact
asymmetry `hyperparameter_tuning.py` already flags for ALS. So **both arms get the identical grid.**

| arm | prose block | long-text block |
|---|---|---|
| A — arctic | `title`+`blurb` **embedded**, weight `p` | `description` **embedded** (pooled), weight `r` |
| B — TF-IDF | `title`,`blurb` TF-IDF at `p/sqrt(2)` each | `reviews` TF-IDF, weight `r` |

`p/sqrt(2)` per role is what makes the two arms' prose blocks carry the *same share of the norm*:
arctic's single block has w²=p², and TF-IDF's two blocks sum to 2·(p/√2)² = p². The baseline sits
at p=√2, r=0.5. `creator`=1.0 and `taxonomy`=0.5 are held fixed in both.

**Cost control.** Vectorizers and embeddings are fit ONCE per arm; a weight change only rescales
pre-computed blocks, so each grid point costs a cheap re-assembly + content cache + 3 λ values.
The search uses a reduced k-grid as a proxy objective; the winners are then re-verified on the FULL
objective so the reported numbers stay comparable to everything already on record.

In [1]:
%%time
import gc
import hashlib
import json
import os
import pickle
import sys
import time

sys.path.insert(0, "..")

import numpy as np
import scipy.sparse as sparse
from sklearn.preprocessing import normalize

from recsys import load, cf, cbhcf, content, item_space, intervention_a as ia, eval as ev

MODEL, DEVICE = "arctic-l-v2.0", "cuda:1"
DATA_PATH = "../data/filtered/books_5core_common.parquet"
META_PATH = "../data/filtered/books_meta_5core_common.parquet"
SPLIT_PARAMS = dict(cold_item_fraction=0.10, cold_val_fraction=0.10)
EMBED_DIR = "../data/embeddings"

SELECTOR, K = "NDCG", 100
K_SEARCH = [0, 5, 20]                 # proxy objective during the grid search
K_FULL = [0, 2, 5, 10, 20]            # the objective everything else is reported on
LAMBDA_SEARCH = [2.0, 3.0, 4.0]       # brackets the warm-guard boundary seen in every run so far
LAMBDA_FULL = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 16.0]
N_SEEDS, N_WARM_EVAL, WARM_TOLERANCE = 2, 10_000, 0.02
ALS_PARAMS = dict(factors=64, regularization=0.01, iterations=20)

PROSE_W = [1.0, float(np.sqrt(2)), 2.0, 3.0]     # sqrt(2) == the current default
LONG_W = [0.0, 0.5, 1.0, 2.0]                    # 0.0 deletes the block; 0.5 == the current default
N_VERIFY = 2                                     # grid points per arm re-run on the full objective


def cache_pickle(name, compute, params=None):
    tag = "" if params is None else "_" + hashlib.md5(repr(params).encode()).hexdigest()[:8]
    path = f"../data/cache/{name}{tag}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    val = compute()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(val, f, protocol=pickle.HIGHEST_PROTOCOL)
    return val


dataset = cache_pickle("books_dataset", lambda: load.load_dataset(data_path=DATA_PATH, **SPLIT_PARAMS),
                       params=load.load_params_fingerprint(DATA_PATH, **SPLIT_PARAMS))
FP = load.dataset_fingerprint(dataset)
val = dataset.cold_val

als_seeds = cache_pickle(
    "als_seeds_tune",
    lambda: [cf.ALSModel(random_state=s, **ALS_PARAMS).fit(dataset.ref_train) for s in range(N_SEEDS)],
    params=(FP, tuple(sorted(ALS_PARAMS.items())), N_SEEDS))
for m in als_seeds:
    m.prepare_gpu_recommend(val, candidates="warm_cold", device=DEVICE)

warm_item_ids = np.unique(dataset.ref_train.nonzero()[1])
eval_users_val = np.flatnonzero(np.diff(val.test_matrix.tocsr().indptr))
_warm_pool = np.flatnonzero(np.diff(dataset.ref_val.tocsr().indptr))
warm_eval_users = np.sort(np.random.default_rng(0).choice(
    _warm_pool, min(N_WARM_EVAL, len(_warm_pool)), replace=False))
_keep = np.zeros(dataset.n_users, dtype=bool)
_keep[warm_eval_users] = True
ref_val_sample = dataset.ref_val.multiply(_keep[:, None]).tocsr()
ref_val_sample.eliminate_zeros()
cache_users = np.union1d(eval_users_val, warm_eval_users)
print(f"cold_val {len(val.cold_item_ids):,} items   eval users {len(eval_users_val):,}   "
      f"cache covers {len(cache_users):,}")

/home/freya/miniforge3/envs/699-gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/freya/miniforge3/envs/699-gpu/lib/python3.12/site-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of '/home/freya/miniforge3/envs/699-gpu/lib/python3.12/site-packages/implicit/gpu/_cuda.so: undefined symbol: _ZN3rmm13device_bufferC1EmNS_16cuda_stream_viewENS_6detail23cccl_async_resource_refIN4cuda2mr3__412resource_refIJNS6_17device_accessibleEEEEEE'
  warnings.warn(
<timed exec>:41: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to acc

cold_val 2,727 items   eval users 12,264   cache covers 21,899
CPU times: user 2.97 s, sys: 678 ms, total: 3.64 s
Wall time: 1.83 s


## Fit once, reweight many times

The expensive parts — the TF-IDF vectorizers over 487,790 documents, and reading the embedding
blocks — depend only on the WARM item set, not on any weight. Doing them once turns each grid
point from a ~5 minute rebuild into a rescale.

In [2]:
%%time
docs = content.load_item_documents(META_PATH, dataset, field_map=content.BOOKS_FIELD_MAP)
space_all = content.fit_content_space(docs, warm_item_ids, weights=None, min_df=2, verbose=True)

# Per-role blocks, unweighted and row-unit-norm. Reweighting is then just a scalar multiply.
ROLE_BLOCKS = {}
for role, vec in space_all.vectorizers.items():
    ROLE_BLOCKS[role] = vec.transform(docs[role]).astype(np.float32)
print("per-role TF-IDF blocks:", {r: b.shape[1] for r, b in ROLE_BLOCKS.items()})
del docs
gc.collect()

DENSE_TEXT = ia.item_embeddings(dataset, MODEL, "books", EMBED_DIR, verbose=True)
_groups = ia.description_embeddings(dataset, MODEL, "books", EMBED_DIR, verbose=True)
DENSE_DESC = np.zeros_like(_groups[list(_groups)[0]])
for g in _groups:
    DENSE_DESC += _groups[g]
_n = np.sqrt(np.einsum("ij,ij->i", DENSE_DESC, DENSE_DESC))
DENSE_DESC /= np.maximum(_n, 1e-12)[:, None]
del _groups
gc.collect()
print(f"dense text {DENSE_TEXT.shape}   dense description (pooled) {DENSE_DESC.shape}")


def space_arm_a(p, r):
    """arctic: embedded prose at weight p, embedded description at weight r, TF-IDF entities."""
    blocks = [DENSE_TEXT * np.float32(p)]
    if r > 0:
        blocks.append(DENSE_DESC * np.float32(r))
    dense = np.hstack(blocks).astype(np.float32) if len(blocks) > 1 else blocks[0].copy()
    sp = sparse.hstack([ROLE_BLOCKS["creator"] * 1.0, ROLE_BLOCKS["taxonomy"] * 0.5],
                       format="csr", dtype=np.float32)
    return item_space.BlockItemSpace.from_blocks(dense, sp, dense_weight=1.0, own_dense=True)


def space_arm_b(p, r):
    """TF-IDF: title and blurb at p/sqrt(2) each -- so the prose block carries w^2 = p^2, exactly the
    share arm A's single block carries at weight p -- plus reviews at r, and the same entities."""
    per = p / np.sqrt(2.0)
    parts = [ROLE_BLOCKS["title"] * per, ROLE_BLOCKS["blurb"] * per,
             ROLE_BLOCKS["creator"] * 1.0, ROLE_BLOCKS["taxonomy"] * 0.5]
    if r > 0:
        parts.append(ROLE_BLOCKS["reviews"] * r)
    X = sparse.hstack(parts, format="csr", dtype=np.float32)
    return item_space.SparseItemSpace(normalize(X, norm="l2", axis=1, copy=False))

role          kind     vocab     fit nnz  nonempty(fit)  weight
title         text    35,822   1,274,113         99.38%    1.00
creator     entity    35,203     198,741         78.51%    1.00
taxonomy    entity       631     719,789         98.58%    0.50
blurb         text   131,015  18,432,005         98.72%    1.00
reviews       text   287,756  55,108,414         77.98%    0.50
TOTAL                490,427
per-role TF-IDF blocks: {'title': 35822, 'creator': 35203, 'taxonomy': 631, 'blurb': 131015, 'reviews': 287756}
[intervention_a] arctic-l-v2.0: 487,790/487,790 items embedded (dim 1024)
[intervention_a] author_bio         331,702/487,790 items (68.0%)
[intervention_a] editorial_review   234,799/487,790 items (48.1%)
[intervention_a] jacket_copy        155,065/487,790 items (31.8%)
dense text (487790, 1024)   dense description (pooled) (487790, 1024)
CPU times: user 3min 24s, sys: 15.7 s, total: 3min 39s
Wall time: 3min 40s


## The grid

Both arms, identical `(p, r)` points, identical λ set, identical everything else. Checkpointed per
grid point — this is ~5 h and the session has lost long runs before.

In [3]:
%%time
CKPT = "../outputs/intervention_a_weight_sweep_checkpoint.json"
CKPT_KEY = {"fingerprint": list(FP), "model": MODEL, "prose_w": PROSE_W, "long_w": LONG_W,
            "lambda_search": LAMBDA_SEARCH, "k_search": K_SEARCH, "seeds": N_SEEDS,
            "warm_tolerance": WARM_TOLERANCE,
            # The dataset fingerprint cannot see a change in how content.py TOKENIZES the metadata:
            # same split, same parquet, different item vectors. Without this entry the creator
            # role-suffix fix would have resumed a 32-point grid computed under the OLD tokens and
            # rewritten it as freshly tuned. Vocabulary size and nnz per role move whenever the
            # tokenizer does, so keying on them invalidates the checkpoint automatically.
            "role_blocks": {r: [int(b.shape[1]), int(b.nnz)] for r, b in sorted(ROLE_BLOCKS.items())}}
grid = {}
if os.path.exists(CKPT):
    b = json.load(open(CKPT))
    if b.get("key") == CKPT_KEY:
        grid = b["results"]
        print(f"resuming: {len(grid)}/{2 * len(PROSE_W) * len(LONG_W)} grid points done")
    else:
        print("checkpoint found but its key does not match this run -- recomputing the grid. "
              "(Differing entries: "
              + ", ".join(sorted(k for k in CKPT_KEY if b.get("key", {}).get(k) != CKPT_KEY[k]))
              + ")")


def save_ckpt():
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    json.dump({"key": CKPT_KEY, "results": grid}, open(CKPT, "w"), default=float)


def evaluate(space, lambdas, k_levels):
    """One space, a few lambdas -> per-lambda (objective, warm). Content cache built once."""
    base = cbhcf.wrap_seeds(als_seeds, dataset.ref_train, space, cache_users, content_weight=1.0,
                            gpu_device=DEVICE, build_mode_b=False, cache_path=None, verbose=False)
    out = []
    for lam in lambdas:
        ms = [m.with_content_weight(lam) for m in base]
        curve, _ = ev.sweep_mode_a_cached(ms, val, k_levels, K=K, verbose=False, with_auc=False)
        ceil = ev.ceiling_reference(ms, val, K=K)
        warm = ev.mode_a_metrics_at_k(ms[0], dataset.ref_train, ref_val_sample, K=K)
        obj = float(np.nanmean(list(curve[SELECTOR]["mean"]) + [ceil["mean"][SELECTOR]]))
        out.append({"lambda": lam, "objective": obj, "warm": float(warm[SELECTOR])})
    del base
    gc.collect()
    return out


for arm, builder in (("A_arctic", space_arm_a), ("B_tfidf", space_arm_b)):
    for p in PROSE_W:
        for r in LONG_W:
            key = f"{arm}|p={p:.4f}|r={r:.2f}"
            if key in grid:
                continue
            t0 = time.perf_counter()
            sp = builder(p, r)
            rows = evaluate(sp, LAMBDA_SEARCH, K_SEARCH)
            best = max(rows, key=lambda x: x["objective"])
            grid[key] = {"arm": arm, "p": p, "r": r, "rows": rows,
                         "best_objective": best["objective"], "best_lambda": best["lambda"],
                         "best_warm": best["warm"],
                         "features": int(getattr(sp, "n_features", 0)),
                         "minutes": (time.perf_counter() - t0) / 60}
            print(f"{arm:<9} p={p:<6.3f} r={r:<4.2f}  obj {best['objective']:.5f}  "
                  f"lam {best['lambda']:g}  warm {best['warm']:.5f}  "
                  f"({grid[key]['minutes']:.1f}m)", flush=True)
            save_ckpt()
            del sp
            gc.collect()

resuming: 32/32 grid points done
CPU times: user 1.44 ms, sys: 114 μs, total: 1.55 ms
Wall time: 1.18 ms


## Search results

The row that answers the description question is **r=0** — the block deleted entirely. If it
matches r=0.5, the field contributes nothing at any weight and the null result is firm.

In [4]:
%%time
for arm in ("A_arctic", "B_tfidf"):
    print(f"\n=== {arm} — proxy objective (k={K_SEARCH}) ===")
    corner = "p vs r"
    print(f"{corner:>8}" + "".join(f"{r:>10.2f}" for r in LONG_W))
    for p in PROSE_W:
        cells = []
        for r in LONG_W:
            g = grid.get(f"{arm}|p={p:.4f}|r={r:.2f}")
            cells.append(f"{g['best_objective']:>10.5f}" if g else f"{'-':>10}")
        star = " *" if abs(p - np.sqrt(2)) < 1e-6 else "  "
        print(f"{p:>8.3f}" + "".join(cells) + star)
    print("  (* = current default prose weight; r=0.50 is the current default long-text weight)")


=== A_arctic — proxy objective (k=[0, 5, 20]) ===
  p vs r      0.00      0.50      1.00      2.00
   1.000   0.03688   0.03842   0.03889   0.03669  
   1.414   0.03752   0.03875   0.03957   0.03787 *
   2.000   0.03740   0.03798   0.03916   0.03836  
   3.000   0.03564   0.03608   0.03687   0.03794  
  (* = current default prose weight; r=0.50 is the current default long-text weight)

=== B_tfidf — proxy objective (k=[0, 5, 20]) ===
  p vs r      0.00      0.50      1.00      2.00
   1.000   0.04513   0.04607   0.04584   0.04397  
   1.414   0.04668   0.04721   0.04745   0.04614 *
   2.000   0.04580   0.04636   0.04702   0.04689  
   3.000   0.04263   0.04309   0.04407   0.04541  
  (* = current default prose weight; r=0.50 is the current default long-text weight)
CPU times: user 301 μs, sys: 0 ns, total: 301 μs
Wall time: 296 μs


## Verify the winners on the full objective

The grid used a reduced k-set for speed. The top points per arm are re-run on the full objective
and full λ grid, so the headline numbers are directly comparable to the six-encoder table, the
description variants, and `cbhcf.coldval_at_selected_lambda`.

In [5]:
%%time
VCK = "../outputs/intervention_a_weight_verify.json"
# Keyed with CKPT_KEY, for the reason recorded above it: `if key in verified: continue` alone would
# skip all four points on a re-run after a tokenizer change and republish the old numbers. The old
# flat-dict format has no "key", so it fails this check and is recomputed -- self-migrating.
_vck = json.load(open(VCK)) if os.path.exists(VCK) else {}
verified = _vck.get("results", {}) if _vck.get("key") == CKPT_KEY else {}
if _vck and not verified:
    print("verify file found but its key does not match this run -- re-verifying from scratch.")
for arm, builder in (("A_arctic", space_arm_a), ("B_tfidf", space_arm_b)):
    pts = sorted([g for g in grid.values() if g["arm"] == arm],
                 key=lambda g: -g["best_objective"])[:N_VERIFY]
    for g in pts:
        key = f"{arm}|p={g['p']:.4f}|r={g['r']:.2f}"
        if key in verified:
            continue
        t0 = time.perf_counter()
        rows = evaluate(builder(g["p"], g["r"]), LAMBDA_FULL, K_FULL)
        ref = max(x["warm"] for x in rows) * (1 - WARM_TOLERANCE)
        feas = [x for x in rows if x["warm"] >= ref] or rows
        best = max(feas, key=lambda x: x["objective"])
        verified[key] = {**{k: g[k] for k in ("arm", "p", "r")}, **best,
                         "warm_floor": ref, "minutes": (time.perf_counter() - t0) / 60}
        print(f"VERIFIED {key}: obj {best['objective']:.5f} at lambda {best['lambda']:g} "
              f"(warm {best['warm']:.5f})  [{verified[key]['minutes']:.1f}m]", flush=True)
        json.dump({"key": CKPT_KEY, "results": verified}, open(VCK, "w"), indent=2, default=float)

print(f"\n{'point':<28}{'obj (full)':>12}{'lambda':>8}{'warm':>10}")
for k, v in sorted(verified.items(), key=lambda kv: -kv[1]["objective"]):
    print(f"{k:<28}{v['objective']:>12.5f}{v['lambda']:>8g}{v['warm']:>10.5f}")

hp = json.load(open("../outputs/hyperparams.json"))
base_a = hp.get("intervention_a", {}).get("per_model", {}).get(MODEL, {}).get("best_objective")
base_b = hp.get("cbhcf", {}).get("coldval_at_selected_lambda", {}).get("objective")
print(f"\nuntuned references: arctic {base_a}   TF-IDF {base_b}")

hp["intervention_a_weight_sweep"] = {
    "model": MODEL, "prose_w": PROSE_W, "long_w": LONG_W, "lambda_search": LAMBDA_SEARCH,
    "k_search": K_SEARCH, "k_full": K_FULL, "seeds": N_SEEDS,
    "note": ("both arms swept on the identical grid so neither has a tuning advantage; arm B's "
             "title/blurb are set to p/sqrt(2) each so its prose block carries the same share of "
             "the item norm as arm A's single block at weight p"),
    "grid": grid, "verified": verified, "dataset_fingerprint": list(FP),
    "role_blocks": CKPT_KEY["role_blocks"],   # provenance: which tokenization these numbers describe
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
json.dump(hp, open("../outputs/hyperparams.json", "w"), indent=2)
print("wrote ../outputs/hyperparams.json -> intervention_a_weight_sweep")


point                         obj (full)  lambda      warm
B_tfidf|p=1.4142|r=1.00          0.04626       3   0.05354
B_tfidf|p=1.4142|r=0.50          0.04478       2   0.05360
A_arctic|p=2.0000|r=1.00         0.03852       4   0.05151
A_arctic|p=1.4142|r=1.00         0.03804       3   0.05122

untuned references: arctic 0.03711975866688272   TF-IDF 0.04551841215442066
wrote ../outputs/hyperparams.json -> intervention_a_weight_sweep
CPU times: user 4.83 ms, sys: 66 μs, total: 4.9 ms
Wall time: 4.6 ms


In [6]:
def evaluate(space, lambdas, k_levels, with_auc=False):
    base = cbhcf.wrap_seeds(als_seeds, dataset.ref_train, space, cache_users, content_weight=1.0,
                            gpu_device=DEVICE, build_mode_b=False, cache_path=None, verbose=False)
    out = []
    for lam in lambdas:
        ms = [m.with_content_weight(lam) for m in base]
        curve, _ = ev.sweep_mode_a_cached(ms, val, k_levels, K=K, verbose=False, with_auc=with_auc)
        ceil = ev.ceiling_reference(ms, val, K=K)
        warm = ev.mode_a_metrics_at_k(ms[0], dataset.ref_train, ref_val_sample, K=K)
        obj = float(np.nanmean(list(curve[SELECTOR]["mean"]) + [ceil["mean"][SELECTOR]]))
        row = {"lambda": lam, "objective": obj, "warm": float(warm[SELECTOR])}
        if with_auc:
            row["auc_by_k"] = [float(v) for v in curve["AUC"]["mean"]]
        out.append(row)
    del base
    gc.collect()
    return out

In [7]:
%%time
# Duplicate of the cell above, but calling the `evaluate` redefined in the previous cell so the
# verified records also carry `auc_by_k`. Only one of the two needs to run: whichever goes first
# populates `verified`, and the other then skips every point. Run THIS one if you want AUC.
VCK = "../outputs/intervention_a_weight_verify.json"
_vck = json.load(open(VCK)) if os.path.exists(VCK) else {}
verified = _vck.get("results", {}) if _vck.get("key") == CKPT_KEY else {}
if _vck and not verified:
    print("verify file found but its key does not match this run -- re-verifying from scratch.")
for arm, builder in (("A_arctic", space_arm_a), ("B_tfidf", space_arm_b)):
    pts = sorted([g for g in grid.values() if g["arm"] == arm],
                 key=lambda g: -g["best_objective"])[:N_VERIFY]
    for g in pts:
        key = f"{arm}|p={g['p']:.4f}|r={g['r']:.2f}"
        if key in verified:
            continue
        t0 = time.perf_counter()
        rows = evaluate(builder(g["p"], g["r"]), LAMBDA_FULL, K_FULL, with_auc=True)
        ref = max(x["warm"] for x in rows) * (1 - WARM_TOLERANCE)
        feas = [x for x in rows if x["warm"] >= ref] or rows
        best = max(feas, key=lambda x: x["objective"])
        verified[key] = {**{k: g[k] for k in ("arm", "p", "r")}, **best,
                         "warm_floor": ref, "minutes": (time.perf_counter() - t0) / 60}
        print(f"VERIFIED {key}: obj {best['objective']:.5f} at lambda {best['lambda']:g} "
              f"(warm {best['warm']:.5f})  [{verified[key]['minutes']:.1f}m]", flush=True)
        json.dump({"key": CKPT_KEY, "results": verified}, open(VCK, "w"), indent=2, default=float)

print(f"\n{'point':<28}{'obj (full)':>12}{'lambda':>8}{'warm':>10}")
for k, v in sorted(verified.items(), key=lambda kv: -kv[1]["objective"]):
    print(f"{k:<28}{v['objective']:>12.5f}{v['lambda']:>8g}{v['warm']:>10.5f}")

hp = json.load(open("../outputs/hyperparams.json"))
base_a = hp.get("intervention_a", {}).get("per_model", {}).get(MODEL, {}).get("best_objective")
base_b = hp.get("cbhcf", {}).get("coldval_at_selected_lambda", {}).get("objective")
print(f"\nuntuned references: arctic {base_a}   TF-IDF {base_b}")

hp["intervention_a_weight_sweep"] = {
    "model": MODEL, "prose_w": PROSE_W, "long_w": LONG_W, "lambda_search": LAMBDA_SEARCH,
    "k_search": K_SEARCH, "k_full": K_FULL, "seeds": N_SEEDS,
    "note": ("both arms swept on the identical grid so neither has a tuning advantage; arm B's "
             "title/blurb are set to p/sqrt(2) each so its prose block carries the same share of "
             "the item norm as arm A's single block at weight p"),
    "grid": grid, "verified": verified, "dataset_fingerprint": list(FP),
    "role_blocks": CKPT_KEY["role_blocks"],   # provenance: which tokenization these numbers describe
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
json.dump(hp, open("../outputs/hyperparams.json", "w"), indent=2)
print("wrote ../outputs/hyperparams.json -> intervention_a_weight_sweep")


point                         obj (full)  lambda      warm
B_tfidf|p=1.4142|r=1.00          0.04626       3   0.05354
B_tfidf|p=1.4142|r=0.50          0.04478       2   0.05360
A_arctic|p=2.0000|r=1.00         0.03852       4   0.05151
A_arctic|p=1.4142|r=1.00         0.03804       3   0.05122

untuned references: arctic 0.03711975866688272   TF-IDF 0.04551841215442066
wrote ../outputs/hyperparams.json -> intervention_a_weight_sweep
CPU times: user 3.71 ms, sys: 0 ns, total: 3.71 ms
Wall time: 3.22 ms


In [8]:
%%time
# The key `steel_thread.ipynb` and `intervention_b_coldllm.ipynb` actually read is
# `steel_thread_config` -- and nothing used to write it. It was transcribed by hand from the table
# above, so a re-tune silently changed nothing: this notebook rewrote its own section, the steel
# thread kept reading the stale config, and the split fingerprint could not object because the
# split had not moved. Deriving it here, from `verified`, closes the loop.
#
# The mapping lives in `recsys.steel_config` -- library code, next to the reader the steel thread
# uses, so both directions of this artifact have one definition and can be checked outside a
# notebook: `python -m recsys.steel_config --check` re-derives it and diffs against what is stored.
from recsys import steel_config

_stc = steel_config.write("../outputs/hyperparams.json")   # backs up, derives, stamps, writes

_c, _a = _stc["cbhcf"], _stc["intervention_a"]
print("wrote ../outputs/hyperparams.json -> steel_thread_config")
print(f"  cbhcf           lambda={_c['content_weight']:g}   "
      + "  ".join(f"{k}={v:g}" for k, v in _c["field_weights"].items()))
print(f"  intervention_a  lambda={_a['content_weight']:g}   text_weight={_a['text_weight']:g}   "
      f"description_weight={_a['description_weight']:g}")
print(f"  coldval         cbhcf {_c['coldval_objective']:.5f}   "
      f"intervention_a {_a['coldval_objective']:.5f}")
print("\nsteel_thread.ipynb can now run without any hand edit to hyperparams.json.")

wrote ../outputs/hyperparams.json -> steel_thread_config
  cbhcf           lambda=3   title=1  creator=1  taxonomy=0.5  blurb=1  reviews=1
  intervention_a  lambda=4   text_weight=2   description_weight=1
  coldval         cbhcf 0.04626   intervention_a 0.03852

steel_thread.ipynb can now run without any hand edit to hyperparams.json.
CPU times: user 6.58 ms, sys: 0 ns, total: 6.58 ms
Wall time: 6.08 ms
